# GitHub API

_Part of the **Decay Before Archival** project — see `README.md` for full project context, setup instructions, and links to the other notebooks (`deps_dev.ipynb`, `scorecard.ipynb`, `gh_archive.ipynb`, `pypi.ipynb`, `github_api.ipynb`, `other_sources.ipynb`)._

In [ ]:
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()  # picks up GITHUB_TOKEN from .env if you have one (optional, see below)

session = requests.Session()
session.headers.update({
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
    "User-Agent": "decay-before-archival",  # required by the GitHub API
})

token = os.environ.get("GITHUB_TOKEN")  # optional: raises rate limits a lot (5000 req/h vs 60)
if token:
    session.headers["Authorization"] = f"Bearer {token}"

print("authenticated" if token else "unauthenticated — search is limited to 10 requests/min, core API to 60/hour")

## Explore: GitHub Search API

The GitHub REST API fills gaps the BigQuery sources don't cover directly — live repo state (stars, forks, open issues/PRs, archived flag, last push) for repos that may not be in deps.dev/Scorecard yet.

**Planned analysis:** build the candidate pool of **open-source repos with 200-50000 stars created between 2024 and 2025**. One catch: GitHub's search API returns at most **1,000 results per query**, so a single `stars:200..50000` query (~28k matches) can't be paged past item 1,000 — the cell below splits the star range into buckets recursively (bisecting any bucket with >1,000 matches until each leaf fits the cap), then merges and de-duplicates them. GitHub's search API also has no license qualifier, so the open-source filter is applied after fetching — keep only repos with an identifiable SPDX license (`license_spdx`); `NOASSERTION` (custom license) and unlicensed repos are dropped by default. These are young-but-popular projects — the cohort we'll later watch for decay signals (stalled releases, contributor turnover, issue backlog growth) before archival.

**Rate limits:** unauthenticated search allows 10 requests/min and the core API 60/hour; a `GITHUB_TOKEN` in `.env` (free account) raises that to 30 searches/min and 5,000 core req/h. The `gh_get` helper below backs off automatically when it hits a limit.

In [ ]:
# Headers of the most recent response — fetch_all() reads the Link header from here for pagination.
_last_headers = {}

def gh_get(url, params=None):
    """GET with automatic backoff on rate-limit (403/429). Returns parsed JSON."""
    if "session" not in globals():
        raise RuntimeError(
            "GitHub session is not set up — run the first code cell "
            "(requests.Session + GITHUB_TOKEN) before running this notebook's queries."
        )
    while True:
        resp = session.get(url, params=params, timeout=30)
        if resp.status_code in (403, 429):
            reset = int(resp.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait = max(reset - time.time(), 1) + 1
            print(f"rate-limited — sleeping {wait:.0f}s")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        _last_headers.clear()
        _last_headers.update(resp.headers)
        return resp.json()


In [ ]:
# Candidate pool: repos created 2024-2025 with 200-50000 stars, most-starred first
import math
from pathlib import Path

def _repo_root():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "pyproject.toml").exists():
            return cand
    return Path.cwd()

DATA_DIR = _repo_root() / "data"
POOL_CSV = str(DATA_DIR / "pool.csv")
os.makedirs(DATA_DIR, exist_ok=True)

STAR_MIN, STAR_MAX = 200, 50000
CREATED_RANGE = "created:2024-01-01..2025-12-31"
per_page = 100

# GitHub's search API returns at most 1,000 results per query (page*per_page <= 1000),
# so one query can't cover the whole pool. Split the star range into buckets instead:
# probe each bucket; if it has >1000 matches, bisect it until every leaf fits the cap.
FULL_POOL = True    # False -> explore mode: only the top max_pages*per_page repos
max_pages = 3       # only used when FULL_POOL = False
REFRESH_POOL = False  # True -> re-run the (slow) bucketed search; otherwise reuse pool.csv

SEARCH_URL = "https://api.github.com/search/repositories"

def fetch_bucket(lo, hi):
    """All repos with lo <= stars <= hi created in range; bisects buckets over the 1000 cap."""
    q = f"stars:{lo}..{hi} {CREATED_RANGE}"
    time.sleep(2.5)  # stay well under the 30 searches/min authenticated limit
    data = gh_get(SEARCH_URL, params={"q": q, "sort": "stars", "order": "desc",
                                      "per_page": per_page, "page": 1})
    total = data["total_count"]
    if total > 1000:
        mid = (lo + hi) // 2
        if lo == hi or mid <= lo:
            print(f"warning: stars:{lo}..{hi} has {total:,} matches but can't be split further — "
                  f"taking only the top 1,000")
            return data["items"]
        left = fetch_bucket(lo, mid)
        right = fetch_bucket(mid + 1, hi)
        print(f"  bucket stars:{lo}..{hi}: {total:,} matches -> split ({len(left):,} + {len(right):,})")
        return left + right
    items = list(data["items"])
    for page in range(2, math.ceil(total / per_page) + 1):
        time.sleep(2.5)
        data = gh_get(SEARCH_URL, params={"q": q, "sort": "stars", "order": "desc",
                                          "per_page": per_page, "page": page})
        items.extend(data["items"])
    print(f"  bucket stars:{lo}..{hi}: {total:,} repos fetched")
    return items

def build_pool(items):
    """Raw search items -> tidy pool DataFrame (columns, open-source filter, star sort)."""
    df = pd.DataFrame(items).drop_duplicates(subset="full_name")
    df["license_spdx"] = df["license"].apply(lambda L: L["spdx_id"] if L else None)
    df = df[[
        "full_name", "stargazers_count", "forks_count", "open_issues_count",
        "language", "created_at", "pushed_at", "archived", "license_spdx", "description",
    ]].rename(columns={"full_name": "repo_full_name"})
    df["repo_owner"] = df["repo_full_name"].str.split("/").str[0]
    df["repo_name"] = df["repo_full_name"].str.split("/").str[1]
    df["created_at"] = pd.to_datetime(df["created_at"]).dt.date
    df["pushed_at"] = pd.to_datetime(df["pushed_at"]).dt.date

    # GitHub's search API has no license qualifier, so "open source" is filtered here:
    #   license_spdx == None        -> no license file at all (not open source)
    #   license_spdx == "NOASSERTION" -> custom/unrecognized license (may still be OSS)
    OPEN_SOURCE_ONLY = True
    if OPEN_SOURCE_ONLY:
        n_before = len(df)
        df = df[df["license_spdx"].notna()]
        print(f"open-source filter: {n_before} -> {len(df)} rows "
              f"(dropped {n_before - len(df)} without an identifiable license)")

    # buckets are fetched independently, so re-sort globally by stars (most-starred first)
    return df.sort_values("stargazers_count", ascending=False).reset_index(drop=True)

if (not REFRESH_POOL) and os.path.exists(POOL_CSV):
    repos = pd.read_csv(POOL_CSV)
    for col in ["created_at", "pushed_at"]:
        repos[col] = pd.to_datetime(repos[col]).dt.date
    print(f"loaded pool from cache: {len(repos):,} repos ({POOL_CSV})")
else:
    print(f"searching for repos with {STAR_MIN}-{STAR_MAX} stars created in 2024-2025 ...")
    t0 = time.time()
    items = fetch_bucket(STAR_MIN, STAR_MAX)
    if not FULL_POOL:
        items = items[: max_pages * per_page]
    repos = build_pool(items)
    print(f"pool: {len(repos):,} repos in {time.time() - t0:.0f}s")
    repos.to_csv(POOL_CSV, index=False)

repos.head(20)

**Notes:**

_(add observations here)_

In [ ]:
# Quick look at the pool: size, year split, languages, and how many are already archived
repos["created_year"] = pd.to_datetime(repos["created_at"]).dt.year
print(f"rows fetched: {len(repos):,}")
print("\nby creation year:")
print(repos["created_year"].value_counts().sort_index())
print("\ntop languages:")
print(repos["language"].value_counts(dropna=False).head(8))
print(f"\nalready archived: {int(repos['archived'].sum())}")

**Notes:**

_(add observations here)_

In [ ]:
# Per-repo detail for one sample repo: contributors + open issues (bus factor, issue backlog)
sample_repo = repos.iloc[0]["repo_full_name"]  # most-starred in the pool; swap in any "owner/name"
print(f"exploring {sample_repo}")

# Contributors (top page of 100 by commit count)
contributors = gh_get(f"https://api.github.com/repos/{sample_repo}/contributors", params={"per_page": 100})
contrib_df = pd.DataFrame(contributors)[["login", "contributions"]]

# Open issues (includes PRs — GitHub counts them together in open_issues_count)
issues = gh_get(f"https://api.github.com/repos/{sample_repo}/issues", params={"per_page": 100, "state": "open"})
issue_df = pd.DataFrame(issues)[["number", "title", "created_at", "pull_request"]]
issue_df["is_pr"] = issue_df["pull_request"].notna()

total_commits = contrib_df["contributions"].sum()
top_share = contrib_df.iloc[0]["contributions"] / total_commits if total_commits else float("nan")
print(f"contributors (top {len(contrib_df)}): {len(contrib_df)}, top contributor share: {top_share:.1%}")
print(f"open issues/PRs (first 100): {len(issue_df)}, of which PRs: {int(issue_df['is_pr'].sum())}")
contrib_df.head(10)

**Notes:**

_(add observations here)_

## Collecting the Data Collection Plan metrics

The [Data Collection Plan](../Data%20Collection%20Plan%20Template.csv) assigns these metrics to the **GitHub API** source; this section collects as many of them as the REST API allows, per repo in the pool above:

| Plan metric | How it's collected here |
|---|---|
| Merged PR counts | search `repo:{repo} is:pr merged:>={since}` → `total_count` (one call, no pagination) |
| Opened Issues | search `repo:{repo} is:issue created:>={since}` → `total_count` (`is:issue` excludes PRs) |
| Opened PRs | search `repo:{repo} is:pr created:>={since}` → `total_count` |
| Resolved Issues | search `repo:{repo} is:issue closed:>={since}` → `total_count` (PR merges counted separately) |
| Time Since last Commit | `GET /repos/{repo}/commits?per_page=1` (default branch, newest first) — days between that commit and today. Note: counts bot commits too; the plan says *human* commits — flag for refinement |
| Time Since last release | `GET /repos/{repo}/releases?per_page=1` — days since the latest published release (empty list → never released) |
| Bus Factor | `GET /repos/{repo}/contributors?per_page=100` — smallest number of top contributors whose combined commits reach 50% of total. Capped at the first 100 contributors; bot accounts (`[bot]`) are excluded to approximate *human* contributors |

**Also captured for free from the pool query:** `Stars` and `Forks` (current totals — the plan's *new during window* version needs a time series, see below), plus `open_issues_count`, `archived`, `license_spdx` as context columns.

**Not collectible from the GitHub API** (other sources own these): Commit Frequency (deps.dev), Downloads (PyPI/npm registries), Criticality/Scorecard/Maintained/Code-Review/CI-Tests scores (OpenSSF Scorecard), Number of dependent projects (deps.dev).

**Window:** the plan says "during the observation period" — `WINDOW_DAYS` below sets a trailing window for all rate-based counts (opened/closed issues, merged PRs). Set it to match whatever the group decides the observation period is. The four activity counts come from the search API's `total_count` (one call each, no pagination — that's what makes this fast on busy repos); search results are index-based and can lag the live REST API by a few minutes, so very active repos may differ by ±1–2 from an item-level fetch at the same moment.

**Scale & rate limits:** each repo now costs **4 search + 3 core calls** (the four activity counts come from the search API's `total_count`, so busy repos don't balloon into hundreds of paginated item fetches). The bottleneck is the authenticated *search* quota — 1,800/hour ÷ 4 = **~450 repos/hour** — so the full pool (~25,000 repos after the open-source filter) takes roughly **~2 days** in one run with a `GITHUB_TOKEN`. The pool itself is cached to `data/pool.csv` (set `REFRESH_POOL = True` to re-search), so restarts skip the ~25-min bucketed search. Progress is checkpointed to `data/github_metrics.csv` after every repo: if you get rate-limited mid-run, `gh_get` sleeps until the reset and continues — or just stop and re-run later, it resumes where it left off.

In [ ]:
import datetime as dt

# Observation window for rate-based metrics (plan: "during the observation period")
WINDOW_DAYS = 90
since = (dt.date.today() - dt.timedelta(days=WINDOW_DAYS)).isoformat()

# How many repos to collect this run. len(repos) = the entire pool (the default below);
# set a small number (e.g. 10) while exploring.
# Resumable: rows already in data/github_metrics.csv are skipped, so you can re-run after
# each rate-limit window and it picks up where it stopped.
N_SAMPLE = len(repos)

print(f"window: last {WINDOW_DAYS} days (since {since}) — collecting for up to {N_SAMPLE:,} repos")

In [ ]:
# Path helpers + DATA_DIR are defined in the pool cell above.
METRICS_CSV = str(DATA_DIR / "github_metrics.csv")

# Fast activity counts via the search API's total_count — one call each, no pagination.
# This is what makes collection fast: instead of fetching every issue/PR item on a busy
# repo (hundreds to thousands of pages), we just ask GitHub for the count.
import threading

SEARCH_COUNT_URL = "https://api.github.com/search/issues"
_search_lock = threading.Lock()
_next_search_at = [0.0]

def search_count(query):
    """total_count for an issue/PR search — one call, no pagination.

    Paced to ~29 searches/min (shared across worker threads) so we stay just under the
    authenticated 30/min search limit. This pace is also the run's throughput ceiling:
    4 search calls per repo -> ~450 repos/hour.
    """
    with _search_lock:
        now = time.time()
        if now < _next_search_at[0]:
            time.sleep(_next_search_at[0] - now)
        _next_search_at[0] = max(time.time(), _next_search_at[0]) + 2.1
    data = gh_get(SEARCH_COUNT_URL, params={"q": query, "per_page": 1})
    return data["total_count"]

In [ ]:
def collect_repo(full_name):
    """Collect the plan's GitHub-API metrics for one repo. 4 search + 3 core calls.

    The four activity counts use the *search* API's total_count (one call each, no
    pagination) — that's what made the old paginated version slow on busy repos. The
    other three metrics are single core-API calls.

    Note: search counts are index-based and can lag the live REST API by a few minutes,
    so a very active repo may differ by ±1-2 from a same-moment item-level fetch.
    """
    row = {"repo_full_name": full_name}

    # --- Activity counts in the window, via search total_count (no pagination) ---
    row["opened_issues"]   = search_count(f"repo:{full_name} is:issue created:>={since}")
    row["opened_prs"]      = search_count(f"repo:{full_name} is:pr created:>={since}")
    row["resolved_issues"] = search_count(f"repo:{full_name} is:issue closed:>={since}")
    row["merged_prs"]      = search_count(f"repo:{full_name} is:pr merged:>={since}")

    # --- Time since last commit (newest commit on default branch) ---
    commits = gh_get(f"https://api.github.com/repos/{full_name}/commits", params={"per_page": 1})
    if commits:
        row["last_commit_at"] = pd.to_datetime(commits[0]["commit"]["committer"]["date"]).date()
        row["days_since_last_commit"] = (dt.date.today() - row["last_commit_at"]).days
    else:
        row["last_commit_at"] = None
        row["days_since_last_commit"] = None

    # --- Time since last release (empty list -> never released) ---
    releases = gh_get(f"https://api.github.com/repos/{full_name}/releases", params={"per_page": 1})
    if releases:
        row["last_release_at"] = pd.to_datetime(releases[0]["published_at"]).date()
        row["days_since_last_release"] = (dt.date.today() - row["last_release_at"]).days
    else:
        row["last_release_at"] = None
        row["days_since_last_release"] = None

    # --- Bus factor: min contributors covering >=50% of commits (top 100, bots excluded) ---
    contribs = gh_get(f"https://api.github.com/repos/{full_name}/contributors", params={"per_page": 100})
    contribs = [c for c in contribs if not c["login"].endswith("[bot]")]
    total = sum(c["contributions"] for c in contribs)
    bus, acc = None, 0
    for n, c in enumerate(contribs, start=1):  # already sorted by contributions desc
        acc += c["contributions"]
        if total and acc * 2 >= total:
            bus = n
            break
    row["bus_factor"] = bus
    row["n_contributors_top100"] = len(contribs)
    return row

In [ ]:
# Batch-collect the pool with a small worker pool, resuming from data/github_metrics.csv.
# Throughput is paced by the shared search limiter (~450 repos/hour at 4 searches/repo),
# so the workers mostly overlap their core-API calls; more workers won't go faster than
# the search quota allows.
target = repos.head(N_SAMPLE)  # most-starred first; swap for a random sample later if desired

if os.path.exists(METRICS_CSV):
    done = set(pd.read_csv(METRICS_CSV)["repo_full_name"])
else:
    done = set()

pending = target[~target["repo_full_name"].isin(done)]
print(f"collecting {len(pending)} repos ({len(done)} already in {METRICS_CSV})")

from concurrent.futures import ThreadPoolExecutor, as_completed

WORKERS = 8
_csv_lock = threading.Lock()

def save_row(row):
    """Append one repo's metrics immediately — a rate-limit mid-run never loses finished work."""
    with _csv_lock:
        chunk = pd.DataFrame([row])
        old = pd.read_csv(METRICS_CSV) if os.path.exists(METRICS_CSV) else pd.DataFrame()
        pd.concat([old, chunk], ignore_index=True).to_csv(METRICS_CSV, index=False)

t0 = time.time()
n_done = 0
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futures = {ex.submit(collect_repo, r.repo_full_name): r.repo_full_name
               for r in pending.itertuples(index=False)}
    for fut in as_completed(futures):
        name = futures[fut]
        try:
            save_row(fut.result())
        except Exception as e:  # a single repo failing (e.g. API glitch) shouldn't kill the run
            print(f"  ! {name}: {type(e).__name__}: {e}")
        n_done += 1
        if n_done % 25 == 0 or n_done == len(pending):
            elapsed = time.time() - t0
            rate = n_done / elapsed if elapsed > 0 else float("nan")
            eta_h = (len(pending) - n_done) / rate / 3600 if rate else float("nan")
            print(f"  {n_done}/{len(pending)} done ({rate * 3600:.0f} repos/h, ETA {eta_h:.1f} h, "
                  f"checkpointed to {METRICS_CSV})")

# If you hit the search limit mid-run (429/403), gh_get sleeps until the reset and
# continues — just leave the cell running. Re-running later resumes from the checkpoint.

print(f"finished: {len(pd.read_csv(METRICS_CSV)) if os.path.exists(METRICS_CSV) else 0} rows total in {METRICS_CSV}")

In [ ]:
# Merge collected metrics back onto the pool and inspect
metrics = pd.read_csv(METRICS_CSV)
for col in ["last_commit_at", "last_release_at"]:
    metrics[col] = pd.to_datetime(metrics[col]).dt.date

combined = repos.merge(
    metrics[["repo_full_name", "opened_issues", "opened_prs", "resolved_issues",
             "merged_prs", "days_since_last_commit", "last_commit_at",
             "days_since_last_release", "last_release_at",
             "bus_factor", "n_contributors_top100"]],
    on="repo_full_name", how="left")

print(f"pool: {len(repos)} rows, with collected metrics: {combined['bus_factor'].notna().sum()}")

# decay-at-a-glance view: quiet repos (stale commits/releases) and fragile ones (low bus factor)
combined[[
    "repo_full_name", "stargazers_count", "days_since_last_commit", "days_since_last_release",
    "bus_factor", "opened_issues", "resolved_issues", "merged_prs", "archived",
]].sort_values("days_since_last_commit", ascending=False).head(20)

**Notes:**

_(add observations here)_

## Join key for integration

`repo_owner` + `repo_name` (and the full `full_name`/`html_url`) — same owner/name pair used by GH Archive (`payload.repo.owner.name` / `name`) and OpenSSF Scorecard, so this source joins directly on those. Package-based sources (deps.dev, PyPI) join via the repo's package manifest or a name match in the integration step.

**Export shape:** the pool table (`repos`) plus `data/github_metrics.csv` (one row per repo: `repo_full_name`, opened/closed issue & PR counts, merged PRs, days since last commit/release, bus factor) — merge on `repo_full_name` in `integrate_datasets.ipynb`.